# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ladinski/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Research Paper Findings and Methodology Questions

**Paper:** FlyRank — *The State of AI-Driven SEO in Numbers* (March 2026)

**Purpose:** Critically examine two reported findings and identify methodological questions that could strengthen their interpretation.

### Finding 1 — The Freshness Multiplier (Page 9)

The paper reports that content older than 365 days but refreshed within the previous 30 days has approximately 3.2 times the health score and 57 times the impressions of the comparison group.

This suggests an association between recent content updates and stronger search performance among mature pages.

**My methodology question:**

Were refreshed and non-refreshed pages matched or controlled for pre-update traffic, search intent, content quality, and historical search position?

Content teams may preferentially refresh pages that already have stronger traffic potential. Without accounting for this selection process, some of the observed performance gap could reflect pre-existing differences rather than the effect of refreshing.

**Suggested validation:**

Compare similar pages before and after updates, ideally against matched pages that were not updated during the same period. Report sample sizes, baseline performance, and follow-up outcomes.

**Careful interpretation:** Recently refreshed mature pages were associated with stronger performance in the observed portfolio. The comparison alone does not establish that refreshing caused the full improvement.

### Finding 2 — Click Capture by Position Tier (Page 8)

The paper reports that weighted CTR decreases across search-position tiers:

- Positions 1–3: 0.423%
- Positions 4–10: 0.339%
- Positions 11–20: 0.325%
- Positions 21–50: 0.163%
- Positions 50+: 0.050%

The paper uses total clicks divided by total impressions, which is more appropriate for portfolio-level CTR comparisons than averaging individual page CTR values without weighting.

**My methodology question:**

How much of the observed CTR difference is explained by search position itself, and how much reflects differences in query intent, branded versus non-branded searches, and search-result features?

Pages in different position tiers may serve different types of searches, making their CTR values difficult to compare directly.

**Suggested validation:**

Repeat the analysis within comparable query-intent groups, report the number of pages and impressions in each tier, and assess whether the directional pattern remains consistent.

**Careful interpretation:** Weighted CTR decreased across position tiers in the observed portfolio. This does not establish that improving a page's position or changing its snippet will necessarily produce a particular increase in clicks.

### Connection to My Capstone

Both findings inform my Content Opportunity Scoring project.

Freshness and search position may be useful signals when prioritizing editorial reviews, but they should be evaluated with appropriate controls, honest validation, and explicit limitations.

The goal is to generate defensible review recommendations rather than assume that observed associations imply causal improvements.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My Model Under an Honest Split

**Model:** Random Forest classifier

**Lane:** Refresh / Content Opportunity Scoring

**Dataset:** 30,000 anonymized content-page observations.

### Validation Question

Does the model's apparent ranking performance change when pages are randomly separated compared with holding out entire clients?

### Validation Strategies

**Strategy A — Random page split**

Randomly divide pages into training and testing sets. This allows pages from the same client to appear in both sets, potentially producing optimistic results due to shared client characteristics.

**Strategy B — Client-holdout split**

Use `GroupShuffleSplit` with `client_id` as the grouping variable. All pages belonging to a client must appear exclusively in either training or testing.

### Controlled Comparison

Both strategies will use:

- The same six input features.
- The same Random Forest configuration.
- A 25% test fraction.
- Training-only median imputation.
- Precision@10, Precision@20, and Precision@50.

The two strategies evaluate different test populations, so differences in performance cannot be attributed solely to leakage. However, the comparison helps assess sensitivity to validation design.

The target remains the starter dataset's observed decline category. This experiment tests classification and ranking performance, not future traffic improvement.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
import subprocess
from pathlib import Path

from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit
)
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

repo_dir = Path("/content/flyrank-ml-internship")
csv_path = repo_dir / "data/raw/content_refresh_anonymized.csv"

if not csv_path.exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/Ladinski/flyrank-ml-internship.git",
        str(repo_dir)
    ], check=True)

df = pd.read_csv(csv_path)

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

feature_cols = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return np.asarray(labels)[order[:k]].mean()

def evaluate_split(train_idx, test_idx, split_name):
    train = df.iloc[train_idx]
    test = df.iloc[test_idx]

    model = make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(
            n_estimators=200,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        )
    )

    model.fit(
        train[feature_cols],
        train["is_declining_label"]
    )

    scores = model.predict_proba(
        test[feature_cols]
    )[:, 1]

    labels = test["is_declining_label"].to_numpy()

    return {
        "Split": split_name,
        "Train pages": len(train),
        "Test pages": len(test),
        "Client overlap": len(
            set(train["client_id"]) &
            set(test["client_id"])
        ),
        "Precision@10": precision_at_k(scores, labels, 10),
        "Precision@20": precision_at_k(scores, labels, 20),
        "Precision@50": precision_at_k(scores, labels, 50)
    }

# A. Random page split
all_idx = np.arange(len(df))

random_train, random_test = train_test_split(
    all_idx,
    test_size=0.25,
    random_state=42,
    stratify=df["is_declining_label"]
)

# B. Client-holdout split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

group_train, group_test = next(
    splitter.split(
        df,
        df["is_declining_label"],
        groups=df["client_id"]
    )
)

comparison_df = pd.DataFrame([
    evaluate_split(random_train, random_test, "Random page split"),
    evaluate_split(group_train, group_test, "Client-holdout")
])

display(comparison_df.round(3))

assert set(df.iloc[group_train]["client_id"]).isdisjoint(
    set(df.iloc[group_test]["client_id"])
)

print("Grouped validation check passed.")


,Split,Train pages,Test pages,Client overlap,Precision@10,Precision@20,Precision@50
0,Random page split,22500,7500,31,0.8,0.80,0.88
1,Client-holdout,22885,7115,0,0.6,0.65,0.70


Grouped validation check passed.


### Validation Comparison — Observed Results

I evaluated the same Random Forest configuration using two validation strategies on the 30,000-page anonymized starter dataset.

| Metric | Random page split | Client-holdout |
|---|---:|---:|
| Precision@10 | 0.800 | 0.600 |
| Precision@20 | 0.800 | 0.650 |
| Precision@50 | 0.880 | 0.700 |
| Overlapping clients | 31 | 0 |

**Observation:** The random page split produced higher ranking metrics, but pages from 31 clients appeared in both training and testing.

Under client-holdout validation, Precision@50 decreased from 0.880 to 0.700.

**Interpretation:** The model performs less strongly when evaluated on entirely unseen clients. The random split may benefit from shared client-specific patterns, although the different test populations also contribute uncertainty.

**Decision:** Client-holdout validation is the more appropriate primary evaluation for the goal of applying the model to previously unseen clients.

**Limitation:** This is one split per strategy, not a repeated-validation experiment. The observed difference should not be interpreted as a precise estimate of generalization loss. Neither split independently establishes future-period predictive validity.

## 3. Leakage Audit and Failure Analysis

### Feature Leakage Audit

I reviewed the six features used by my Week 5 Random Forest model.

The following fields were explicitly excluded:

- `trend_direction`: Source of the observed decline label.
- `trend_pct`: Directly related to the target definition.
- `is_declining_label`: The target itself.
- `client_id`: Used only for grouped validation.

All six retained features — impressions, average position, CTR, content age, update recency, and word count — require verification that their observation periods precede the predicted outcome.

No directly label-derived columns were present in the model inputs. However, the snapshot-based starter dataset does not establish complete temporal separation between features and outcomes.

Therefore, this audit rules out obvious target-column leakage but does not prove the model is entirely leakage-free.

### Real Failure Examples

The client-holdout Random Forest achieved Precision@50 of 0.700.

Among its 50 highest-ranked recommendations, 35 pages were labeled as declining and 15 were not.

Five examples of false positives were:

| Review rank | Impressions | Position | Predicted risk | Actual label |
|---|---:|---:|---:|---|
| 1 | 3,445 | 39.0 | 0.984 | Not declining |
| 6 | 209 | 20.0 | 0.952 | Not declining |
| 9 | 1,596 | 6.2 | 0.949 | Not declining |
| 10 | 2,004 | 39.1 | 0.948 | Not declining |
| 17 | 1,174 | 13.8 | 0.940 | Not declining |

### Interpretation

The model assigned high decline-risk scores to several pages that were not labeled as declining.

Some false positives had relatively poor search positions, low CTR, or older content. These characteristics may have influenced their rankings, but the examples alone do not establish which features caused each prediction.

The predicted probabilities have not been independently calibrated. Therefore, a score of 0.984 should not be interpreted as proof that a page has a 98.4% real-world probability of future decline.

### Operational Consequences

False positives could lead editors to spend time reviewing pages that do not require intervention.

False negatives could cause genuinely declining pages to receive insufficient attention.

A useful deployment should provide explanations, involve human review, and evaluate whether recommendations lead to measurable improvements.

### Audit Conclusion

Client-holdout validation provides a more relevant test of new-client generalization than random page splitting.

The model showed useful ranking performance on the held-out clients, but high-confidence mistakes, uncertain temporal alignment, and the proxy nature of the target limit the conclusions that can be drawn.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Audit the model input features for target leakage

audit_rows = [
    ("impressions_90d", "Historical visibility", "REVIEW"),
    ("avg_position", "Historical search position", "REVIEW"),
    ("ctr", "Historical click-through rate", "REVIEW"),
    ("content_age_days", "Content age", "REVIEW"),
    ("days_since_last_update", "Update recency", "REVIEW"),
    ("word_count", "Content length", "REVIEW"),
    ("trend_direction", "Source of target label", "EXCLUDE"),
    ("trend_pct", "Directly related to target", "EXCLUDE"),
    ("is_declining_label", "Prediction target", "EXCLUDE"),
    ("client_id", "Validation grouping identifier", "GROUP ONLY")
]

audit_df = pd.DataFrame(
    audit_rows,
    columns=["Feature", "Purpose", "Decision"]
)

display(audit_df)

# Verify no prohibited target-derived fields enter training
prohibited = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "client_id"
}

assert prohibited.isdisjoint(feature_cols)
print("No explicitly prohibited columns in model inputs.")



,Feature,Purpose,Decision
0,impressions_90d,Historical visibility,REVIEW
1,avg_position,Historical search position,REVIEW
2,ctr,Historical click-through rate,REVIEW
3,content_age_days,Content age,REVIEW
4,days_since_last_update,Update recency,REVIEW
5,word_count,Content length,REVIEW
6,trend_direction,Source of target label,EXCLUDE
7,trend_pct,Directly related to target,EXCLUDE
8,is_declining_label,Prediction target,EXCLUDE
9,client_id,Validation grouping identifier,GROUP ONLY


No explicitly prohibited columns in model inputs.


In [3]:

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# Refit the Week 5 model on the grouped training split
audit_model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )
)

audit_model.fit(
    df.iloc[group_train][feature_cols],
    df.iloc[group_train]["is_declining_label"]
)

# Predict on unseen clients
audit_test = df.iloc[group_test].copy()

audit_test["predicted_risk"] = audit_model.predict_proba(
    audit_test[feature_cols]
)[:, 1]

# Rank by predicted decline risk
audit_test = audit_test.sort_values(
    "predicted_risk",
    ascending=False,
    kind="stable"
).copy()

audit_test["review_rank"] = np.arange(
    1, len(audit_test) + 1
)

# Find false positives among the top 50
top50 = audit_test.head(50)

false_positives = top50[
    top50["is_declining_label"] == 0
]

print("Top-50 recommendations:", len(top50))
print("Correct declining pages:",
      (top50["is_declining_label"] == 1).sum())
print("False positives:", len(false_positives))

# Show five real failure examples without identifiers
display(false_positives[
    [
        "review_rank",
        "impressions_90d",
        "avg_position",
        "ctr",
        "content_age_days",
        "days_since_last_update",
        "predicted_risk"
    ]
].head(5).round(3))


Top-50 recommendations: 50
Correct declining pages: 35
False positives: 15


,review_rank,impressions_90d,avg_position,ctr,content_age_days,days_since_last_update,predicted_risk
22526,1,3445,39.0,0.09,280,104,0.984
2357,6,209,20.0,0.00,271,104,0.952
29401,9,1596,6.2,0.31,275,104,0.949
2150,10,2004,39.1,0.00,272,104,0.948
16667,17,1174,13.8,0.43,275,104,0.940


## 4. Research Claim Rewrite

The purpose of this section is to align my conclusions with what the available evidence actually supports.

### Claim 1 — Model Performance

**Overstated:** "My Random Forest can accurately identify which pages need refreshing."

**Evidence-supported:** "On one client-holdout evaluation of the anonymized starter dataset, the Random Forest achieved Precision@50 of 0.700 for identifying pages labeled as declining. This measures ranking performance against a proxy label, not whether refreshing those pages will improve traffic."

### Claim 2 — Generalization

**Overstated:** "The model generalizes reliably to all new clients."

**Evidence-supported:** "The model achieved Precision@50 of 0.880 under a random page split and 0.700 under a client-holdout split. This observed difference suggests validation design materially affects reported performance, although different test populations also contribute to the comparison."

### Claim 3 — Model Confidence

**Overstated:** "A predicted risk score above 90% means a page is almost certain to decline."

**Evidence-supported:** "Several highly ranked non-declining pages received scores above 0.90. Because the probabilities have not been calibrated, these scores should be interpreted as model ranking estimates rather than verified future-decline probabilities."

### Claim 4 — Business Impact

**Overstated:** "Using this model will increase organic traffic."

**Evidence-supported:** "The model may help prioritize pages for editorial investigation, but the current analysis does not measure the causal effect of content updates. Business impact would require follow-up measurements or controlled intervention studies."

### Claim 5 — Leakage

**Overstated:** "The model has no data leakage."

**Evidence-supported:** "The audit found no directly target-derived variables among the six model inputs, and validation separated clients. However, full temporal availability of the starter features has not been verified, so additional time-aware evaluation is necessary."

### Final Research Conclusion

The Random Forest demonstrated promising ranking performance relative to the Week 4 baseline on a single client-holdout split.

However, the results should be treated as preliminary decision-support evidence.

Further work should focus on temporal validation, repeated grouped evaluation, error analysis, and measuring whether editorial interventions produce improvements beyond what would have happened without those interventions.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.